# Unsloth Fine-Tuning Notebook

Bereinigte, Colab-kompatible Version mit zwei Trainingsmodi:
- `finetome`: Beispiel-Dataset von Hugging Face
- `custom`: kleines eigenes Prompt-Dataset


## 0. Optional: Colab-Selbst-Sync

Nur in Colab ausfuehren, wenn dieses Notebook aus `/content/Test-Codex` gestartet wurde.
Die Zelle zieht den neuesten GitHub-Stand und stoppt danach bewusst, damit du das aktualisierte Notebook neu oeffnest.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Say43/Test-Codex.git"
CONTENT_DIR = Path("/content")
REPO_PREFIX = "Test-Codex"
existing_repos = sorted(path for path in CONTENT_DIR.glob(f"{REPO_PREFIX}*") if (path / ".git").exists())
REPO_DIR = existing_repos[0] if existing_repos else CONTENT_DIR / REPO_PREFIX
NOTEBOOK_PATH = REPO_DIR / "colab_finetuning_unsloth.ipynb"

if "COLAB_" not in "".join(os.environ.keys()):
    print("Self-sync skipped: not running in Google Colab.")
elif not (REPO_DIR / ".git").exists():
    clone_target = CONTENT_DIR / REPO_PREFIX
    suffix = 0
    while clone_target.exists():
        suffix += 1
        clone_target = CONTENT_DIR / f"{REPO_PREFIX}-repo-{suffix}"
    print(f"No git repo found in /content. Cloning the repository into {clone_target}.")
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(clone_target)], check=True)
    print(f"Repo cloned. Open this notebook from: {clone_target / 'colab_finetuning_unsloth.ipynb'}")
    print("Sync step finished successfully.")
else:
    def run(cmd):
        print("$", " ".join(cmd))
        subprocess.run(cmd, cwd=REPO_DIR, check=True)

    run(["git", "fetch", "origin"])
    run(["git", "checkout", "main"])
    run(["git", "reset", "--hard", "origin/main"])
    print(f"Repo synced. Open this notebook from: {NOTEBOOK_PATH}")
    print("Sync step finished successfully.")


## 1. Setup

Installiert die benoetigten Pakete fuer Colab und lokale Tests.

## 0.1 Versionscheck in Colab

Zeigt den aktuell geladenen Git-Commit und vergleicht ihn mit `origin/main`.

In [ ]:
import os
import subprocess
from pathlib import Path

CONTENT_DIR = Path("/content")
REPO_PREFIX = "Test-Codex"
existing_repos = sorted(path for path in CONTENT_DIR.glob(f"{REPO_PREFIX}*") if (path / ".git").exists())
REPO_DIR = existing_repos[0] if existing_repos else CONTENT_DIR / REPO_PREFIX

def git_output(args):
    return subprocess.check_output(args, cwd=REPO_DIR, text=True).strip()

if "COLAB_" not in "".join(os.environ.keys()):
    print("Version check skipped: not running in Google Colab.")
elif not (REPO_DIR / ".git").exists():
    print("Version check skipped: no cloned git repo was found yet.")
    print("Run the self-sync cell above once. It will clone the repository for you.")
else:
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    local_head = git_output(["git", "rev-parse", "HEAD"])
    origin_main = git_output(["git", "rev-parse", "origin/main"])
    short_head = git_output(["git", "rev-parse", "--short", "HEAD"])
    branch_name = git_output(["git", "branch", "--show-current"])

    print(f"Branch: {branch_name}")
    print(f"Local HEAD:  {short_head} ({local_head})")
    print(f"origin/main: {origin_main}")

    if local_head == origin_main:
        print("Status: up to date with origin/main")
    else:
        print("Status: NOT up to date with origin/main")
        print("Run the self-sync cell above, then reopen the notebook.")


In [ ]:
%%capture
import os
import re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v == "2.9" else "0.0.32.post2" if v == "2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


## 2. Modell und LoRA vorbereiten

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True
model_name = "unsloth/Llama-3.2-3B-Instruct"

# 4-bit pre-quantized models for faster loading and lower VRAM usage.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",
    "unsloth/Mistral-Small-Instruct-2409",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",
    "unsloth/Llama-3.2-1B-bnb-4bit",
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # token = "hf_...",  # Use for gated models if needed.
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


## 2.1 Trainingsmodus festlegen

Hier wird zentral zwischen den beiden Fine-Tuning-Varianten gewechselt.

- `finetome`: verwendet `mlabonne/FineTome-100k`
- `custom`: verwendet ein Dataset aus der Sammlung unter `datasets/custom/`

Nur diese Zelle aendern, wenn du die Trainingsquelle wechseln willst.

In [ ]:
TRAINING_MODE = "custom"
assert TRAINING_MODE in {"finetome", "custom"}, "TRAINING_MODE muss finetome oder custom sein"

from pathlib import Path

CONTENT_DIR = Path("/content")
REPO_PREFIX = "Test-Codex"
existing_repos = sorted(path for path in CONTENT_DIR.glob(f"{REPO_PREFIX}*") if (path / ".git").exists())
REPO_DIR = existing_repos[0] if existing_repos else CONTENT_DIR / REPO_PREFIX
LOCAL_PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = REPO_DIR if (REPO_DIR / ".git").exists() else LOCAL_PROJECT_ROOT

CUSTOM_DATASETS = {
    "german_concise_style": {
        "path": PROJECT_ROOT / "datasets" / "custom" / "german_concise_style.jsonl",
        "label": "German concise style",
    },
    "template_dataset": {
        "path": PROJECT_ROOT / "datasets" / "custom" / "template_dataset.jsonl",
        "label": "Template dataset",
    },
}
CUSTOM_DATASET_NAME = "german_concise_style"
assert CUSTOM_DATASET_NAME in CUSTOM_DATASETS, "Unbekanntes Custom-Dataset"
selected_custom_dataset = CUSTOM_DATASETS[CUSTOM_DATASET_NAME]

TRAINING_SOURCES = {
    "finetome": "Hugging Face dataset: mlabonne/FineTome-100k",
    "custom": f"Repository dataset: {selected_custom_dataset['path']}",
}

print(f"Projektordner: {PROJECT_ROOT}")
if TRAINING_MODE == "custom" and not selected_custom_dataset["path"].exists():
    print("Hinweis: Das Custom-Dataset liegt nur im Git-Repo.")
    print("Falls du das Notebook als einzelne Datei in Colab geoeffnet hast, fuehre zuerst die Self-Sync-Zelle aus und oeffne danach das Notebook aus /content/Test-Codex neu.")
print(f"Ausgewaehlter Trainingsmodus: {TRAINING_MODE}")
print(f"Trainingsquelle: {TRAINING_SOURCES[TRAINING_MODE]}")
if TRAINING_MODE == "custom":
    print(f"Custom-Dataset-Name: {CUSTOM_DATASET_NAME}")
    print(f"Custom-Dataset-Datei: {selected_custom_dataset['path']}")


## 3. Chat-Template und FineTome-Dataset

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def format_conversations_as_text(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = load_dataset("mlabonne/FineTome-100k", split="train")
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(format_conversations_as_text, batched=True)


## 4. Eigenes Mini-Dataset

Custom-Datasets liegen jetzt als eigene Dateien im Repo unter `datasets/custom/`.
Hier wird nur noch das ausgewaehlte Dataset geladen.

In [ ]:
from datasets import load_dataset

custom_dataset_path = selected_custom_dataset["path"]
assert custom_dataset_path.exists(), (
    f"Custom-Dataset nicht gefunden: {custom_dataset_path}. "
    "Wenn du in Colab arbeitest, fuehre zuerst die Self-Sync-Zelle aus und oeffne danach das Notebook neu aus dem geklonten Repo unter /content/Test-Codex* ."
)

train_ds_raw = load_dataset(
    "json",
    data_files=str(custom_dataset_path),
    split="train",
)
train_ds = train_ds_raw.map(
    format_conversations_as_text,
    batched=True,
    remove_columns=["conversations"],
)

custom_finetune_label = selected_custom_dataset["label"]

print(f"Geladenes Custom-Dataset: {CUSTOM_DATASET_NAME}")
print(f"Datei: {custom_dataset_path}")
print("Train samples:", len(train_ds))
print(train_ds[0]["text"][:400])


## 5. LoRA-Adapter bei Bedarf neu initialisieren

Nur ausfuehren, wenn du bewusst mit frischen Adaptern neu starten willst.

In [ ]:
# Basismodell neu laden, damit bisherige LoRA-Aenderungen verworfen werden.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

model._finetune_label = custom_finetune_label


## 6. Trainer aufsetzen

Stelle hier zwischen `finetome` und `custom` um.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

if TRAINING_MODE == "finetome":
    active_dataset = dataset
    active_output_dir = "outputs_finetome"
    active_training_source = TRAINING_SOURCES["finetome"]
    model._finetune_label = "FineTome-100k"
else:
    active_dataset = train_ds
    active_output_dir = "outputs_custom"
    active_training_source = TRAINING_SOURCES["custom"]
    model._finetune_label = custom_finetune_label

print(f"Training mode aktiv: {TRAINING_MODE}")
print(f"Verwendete Trainingsquelle: {active_training_source}")
print(f"Anzahl Samples: {len(active_dataset)}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=active_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        # num_train_epochs = 1,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=active_output_dir,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)


## 7. Training starten

In [ ]:
trainer_stats = trainer.train()


## 8. Kurzer Inferenztest

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Wie geht es?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=1.5,
    min_p=0.1,
)


## 9. Einfacher Notebook-Chat

In [ ]:
from threading import Thread
from transformers import TextIteratorStreamer

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = None
# SYSTEM_PROMPT = "Du bist ein hilfreicher Assistent. Antworte knapp und korrekt."

GEN_KWARGS = dict(
    max_new_tokens=256,
    use_cache=True,
    temperature=0.8,
    min_p=0.1,
)

def generate_reply(messages, **gen_kwargs):
    """
    messages: list of dicts with role/content pairs.
    Returns the full assistant reply and streams tokens while generating.
    """
    enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    enc = {k: v.to("cuda") for k, v in enc.items()}

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    t = Thread(
        target=model.generate,
        kwargs=dict(
            input_ids=enc["input_ids"],
            attention_mask=enc.get("attention_mask", None),
            streamer=streamer,
            **gen_kwargs,
        ),
    )
    t.start()

    out = []
    for token in streamer:
        print(token, end="", flush=True)
        out.append(token)
    print()

    return "".join(out).strip()

def chat_loop():
    global SYSTEM_PROMPT
    history = []

    print("Chat gestartet. Tippe deine Nachricht und druecke Enter.")
    print("Commands: /exit, /reset, /sys <text>")
    print("-" * 60)

    while True:
        try:
            user = input("Du: ").strip()
        except EOFError:
            break

        if not user:
            continue
        if user.lower() in ("/exit", "/quit"):
            print("Chat beendet.")
            break
        if user.lower() == "/reset":
            history = []
            print("Historie geloescht.")
            continue
        if user.lower().startswith("/sys "):
            SYSTEM_PROMPT = user[5:].strip() or None
            print(f"System prompt gesetzt: {SYSTEM_PROMPT!r}")
            continue

        messages = []
        if SYSTEM_PROMPT:
            messages.append({"role": "system", "content": SYSTEM_PROMPT})
        for u, a in history:
            messages.append({"role": "user", "content": u})
            messages.append({"role": "assistant", "content": a})
        messages.append({"role": "user", "content": user})

        print("Bot: ", end="", flush=True)
        reply = generate_reply(messages, **GEN_KWARGS)
        history.append((user, reply))

chat_loop()


## 10. LoRA speichern

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token="...")
# tokenizer.push_to_hub("your_name/lora_model", token="...")


## 11. Fine-Tuning-Status pruefen

In [ ]:
def print_finetune_status(model):
    print("\n=== Fine-Tuning Status ===")

    is_peft = hasattr(model, "peft_config") and model.peft_config is not None
    print(f"LoRA active: {is_peft}")

    if not is_peft:
        print("-> Model is BASE (no fine-tuning applied)")
        return

    try:
        active_adapters = model.active_adapters
    except Exception:
        active_adapters = "unknown"

    print(f"Active adapter(s): {active_adapters}")

    for name, cfg in model.peft_config.items():
        print(f"\nAdapter name: {name}")
        print(f"  LoRA rank (r): {cfg.r}")
        print(f"  LoRA alpha:   {cfg.lora_alpha}")
        print(f"  Target mods:  {cfg.target_modules}")

    if hasattr(model, "_finetune_label"):
        print(f"\nUser training label: {model._finetune_label}")
    else:
        print("\nUser training label: NOT SET")
        print("-> Dataset origin cannot be inferred automatically.")

    if "TRAINING_MODE" in globals():
        print(f"Configured training mode: {TRAINING_MODE}")
    if "active_training_source" in globals():
        print(f"Active training source: {active_training_source}")

    print("==========================\n")

print_finetune_status(model)
